In [ ]:
%load_ext autoreload
%autoreload 2

import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import polars as pl

from trunx.config import project_root
from trunx.gp3.PG3_model_impl import run_threepg_main
from trunx.gp3.plot_function import create_comparison_dataframe
from trunx.gp3.run_r3pg import run_comparison_r

os.chdir(project_root)

In [ ]:
file_path = "./data/data_sspecies_nothinning.xlsx"
# fig, outputs, df = run_threepg_main(file_path, plot_output=True, r_comparison = True)

In [ ]:
r_df = run_comparison_r(file_path)
r_df = pl.DataFrame(r_df)
species_names = r_df.select("species").unique().to_series().to_list()
fig, outputs = run_threepg_main(file_path, plot_output=True, r_comparison=True)

df = create_comparison_dataframe(r_df, outputs, np.datetime64("2001-01"), species_names)

# # df.head()

In [ ]:
r_df

In [ ]:
file_path = "./data/data_sspecies_nothinning.xlsx"

fig, outputs = run_threepg_main(file_path, plot_output=False, r_comparison=True)

# Ensure Dates column is datetime
df = pd.read_csv("./data/r_python.comparison.csv")
df["Dates"] = pd.to_datetime(df["Dates"])

py_metrics = [
    col for col in df.columns if not col.startswith("r_") and col != "Dates" and col != "species"
]

ncols = 4
n_metrics = len(py_metrics)
nrows = (n_metrics + ncols - 1) // ncols

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(4 * ncols, 4 * nrows), sharex=True)
axes = axes.flatten()

for ax, py_col in zip(axes, py_metrics, strict=False):
    r_col = f"r_{py_col}"
    if r_col in df.columns:
        ax.plot(df["Dates"], df[py_col], label="Python", color="tab:blue")
        ax.plot(df["Dates"], df[r_col], label="R", color="tab:orange", linestyle="--")
        ax.set_title(py_col)
        ax.grid(True)
        ax.legend()

for i in range(len(py_metrics), len(axes)):
    fig.delaxes(axes[i])

plt.tight_layout()
plt.show()

In [ ]:
import datetime as dt

import polars as pl

file_path = os.path.join("./data/", "S_weather_data.xlsx")
origin = dt.datetime(1970, 1, 1)
r_outputs = run_comparison_r(file_path)

r_outputs = (
    pl.DataFrame(r_outputs)
    .with_columns(
        pl.col("date")
        .map_elements(lambda x: origin + dt.timedelta(days=x), return_dtype=pl.Datetime)
        .alias("dates")
    )
    .with_columns(
        pl.col("dates").dt.year().alias("year"), pl.col("dates").dt.month().alias("month")
    )
)

r_outputs.head()